In [7]:
import pandas as pd
import requests
import json
import csv
import gzip
import datetime
from datetime import datetime, timedelta
import pytz
import traceback
from xmljson import yahoo
import numpy as np
from lxml import etree
import lxml
from liquer import *

In [143]:
HEADERS_FOR_JSON = {
    'accept': 'application/json; charset-utf-8',
    'content-type': 'application/json'
}
SEARCH_URL = 'http://distribution.virk.dk/offentliggoerelser/_search'

def offentliggoerelser_raw(cvr):
    data = {
        'query': {
            'bool': {
                'must': [
                    {
                        'term': {
                            'dokumenter.dokumentMimeType': 'xml'
                        }
                    },
                    {
                        'term': {
                            'cvrNummer': cvr
                        }
                    },
                    {
                        'range': {
                            'offentliggoerelsesTidspunkt': {
                                'gt': '2020-04-01T00:00:00.001Z',
                                'lt': '2025-04-01T23:59:59.505Z'
                            }
                        }
                    }
                ],
                'must_not': [],
                'should': []
            }
        },
        'size': 2999
    }

    response = requests.post(SEARCH_URL, data=json.dumps(data),
                             headers=HEADERS_FOR_JSON)
    return response.json()

data = offentliggoerelser_raw(33166966)

def register2df(data):
    """Convert JSON response from offentliggoerelser_raw to DataFrame"""
    df = pd.DataFrame(columns=["_id", "_index", "_score", "cvrNummer"])
    for r in data["hits"]["hits"]:
        d = dict(
            _id=r["_id"],
            _index=r["_index"],
            _score=r["_score"],
            cvrNummer=str(r["_source"]["cvrNummer"]),
            indlaesningsId=r["_source"]["indlaesningsId"],
            indlaesningsTidspunkt=r["_source"]["indlaesningsTidspunkt"],
            offentliggoerelsesTidspunkt=r["_source"]["offentliggoerelsesTidspunkt"],
            offentliggoerelsestype=r["_source"]["offentliggoerelsestype"],
            omgoerelse=r["_source"]["omgoerelse"],
            regNummer=r["_source"]["regNummer"],
            regnskabsperiode_startDato=r["_source"]["regnskab"]["regnskabsperiode"]["startDato"],
            regnskabsperiode_slutDato=r["_source"]["regnskab"]["regnskabsperiode"]["slutDato"],
            sagsNummer=r["_source"]["sagsNummer"],
            sidstOpdateret=r["_source"]["sidstOpdateret"],
        )
        for doc in r["_source"]["dokumenter"]:
            t = {"application/xhtml+xml": "html", "application/pdf": "pdf",
                 "application/xml": "xml"}.get(doc["dokumentMimeType"])
            if t is not None:
                if doc["dokumentType"] == "AARSRAPPORT":
                    d[f"AARSRAPPORT_{t}"] = doc["dokumentUrl"]
        df = pd.concat([df, pd.DataFrame([d])], ignore_index=True)
    return df

register2df(data)

def cvr(cvr, ext="xml"):
    if len(register2df(data)) == 0:
        df = register2df(data)
    else:
        df = register2df(data).sort_values(by=['regnskabsperiode_slutDato'])
    url = list(df.loc[df.cvrNummer.map(str) == str(cvr), f"AARSRAPPORT_{ext}"])
    if len(url):
        try:
            return requests.get(url[-1]).text
        except:
            import traceback
            traceback.print_exc()
            return ""
    else:
        return ""
    
cvrdata = cvr(33166966, ext="xml")

def tojson(xml):
    from xmljson import yahoo
    from lxml import etree
    try: 
        root = etree.fromstring(xml.encode("utf-8"))
    except:
        return {}
    for elem in root.getiterator():
        try:
            tag = etree.QName(elem.tag)
        except: 
            traceback.print_exc()
            continue
        elem.tag = tag.localname
        d = {}
        for key, value in elem.attrib.items():
            nkey = etree.QName(key).localname
            d[nkey] = value
            del elem.attrib[key]
        elem.attrib.update(d)

    d = yahoo.data(root)
    return d["xbrl"]

doc = tojson(cvrdata)

def json2df(doc, keep_multiline_values=False, init=None):
    if not isinstance(init, dict):
        init = {}
    df = pd.DataFrame(columns=["entity", "start_date", "end_date", "context"])
    context = {c["id"]: c for c in doc.get("context", [])}
    cdates = []
    for c in doc.get("context", []):
        period = c.get("period", {})
        cid = c["id"]
        if "instant" in period:
            cdates.append((period["instant"], period["instant"], cid))
        elif "startDate" in period and "endDate" in period:
            cdates.append((period["startDate"], period["endDate"], cid))
        else:
            raise Exception("Ceontext without known dates")

    cdates = sorted(cdates)

    cols = sorted(key for key in doc.keys() if key not in ["context", "unit"])
    for start, end, identifier in cdates:
        c = context[identifier]
        entity = c["entity"]["identifier"]["content"]
        d = dict(init, start_date=start, end_date=end, entity=entity, context=identifier)
        for key in cols:
            keydata = doc[key]
            if isinstance(keydata, list):
                row = [r for r in keydata if r["contextRef"] == identifier]
                if len(row):
                    keydata = row[0]
                else:
                    keydata = None
            if keydata is not None:
                if isinstance(keydata, dict) and "content" in keydata:
                    use = keep_multiline_values or "\n" not in keydata["content"]
                    if use:
                        value = keydata["content"]
                        try:
                            value = float(value)
                            if int(value) == value:
                                value = int(value)
                        except:
                            pass
                    d[key] = value
        df = pd.concat([df, pd.DataFrame([d])], ignore_index=True)
    return df

xmlasdf = json2df(doc)

keep_cols = ['entity', 'start_date', 'end_date', 'ProfitLoss', 'Equity', 'Assets', 'AverageNumberOfEmployees']
new_xmlasdf = xmlasdf[xmlasdf.columns.intersection(keep_cols)]
new_xmlasdf['start_date'] = pd.to_datetime(new_xmlasdf['start_date'], format='%Y-%m-%d', errors='coerce')
latest_date = new_xmlasdf["start_date"].max()

def one_year_before_latest(latest_date):
    latest_date_time = pd.Timestamp(latest_date)
    year = latest_date_time.year
    def is_leap_year(year):
        if (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0):
            return True
        return False
    if is_leap_year(year):
        return 366
    else:
        return 365
    

go_back_one_year = one_year_before_latest(latest_date)
one_year_before_newest = latest_date - timedelta(days=go_back_one_year)
filtered_new_xmlasdf = new_xmlasdf[new_xmlasdf['start_date'] > one_year_before_newest]

df = filtered_new_xmlasdf

def new_dataframe(df):
    new_data = {}
    first_non_nan_ProfitLoss_index = df['ProfitLoss'].first_valid_index()
    new_data['entity'] = df.loc[first_non_nan_ProfitLoss_index, 'entity']
    new_data['start_date'] = df.loc[first_non_nan_ProfitLoss_index, 'start_date']
    new_data['end_date'] = df.loc[first_non_nan_ProfitLoss_index, 'end_date']
    new_data['ProfitLoss'] = df.loc[first_non_nan_ProfitLoss_index, 'ProfitLoss']
    Assets_value = df['Assets'].dropna().iloc[0]
    new_data['Assets'] = Assets_value
    new_data['Equity'] = df.loc[df['Assets'] == Assets_value, 'Equity'].values[0]

    if 'AverageNumberOfEmployees' in df.columns:
        average_num_employees_value = df['AverageNumberOfEmployees'].dropna().iloc[0] if df['AverageNumberOfEmployees'].notna().sum() > 0 else 0
        new_data['AverageNumberOfEmployees'] = average_num_employees_value
    else:
        new_data['AverageNumberOfEmployees'] = 0

    column_order = ['entity', 'start_date', 'end_date', 'ProfitLoss', 'Assets', 'Equity', 'AverageNumberOfEmployees']
    new_df = pd.DataFrame([new_data], columns=column_order)

    return new_df


def empty_df(cvr: int):
    columns = ['entity', 'start date', 'end date', 'PnL', 'Assets', 'Equity', 'Number of employees']
    data = {
        'entity': [str(cvr)],
        'start date': [np.nan],
        'end date': [np.nan],
        'PnL': [np.nan],
        'Assets': [np.nan],
        'Equity': [np.nan],
        'Number of employees': [np.nan]
    }
    empty_df = pd.DataFrame(data=data, columns=columns)
    return empty_df

C:\Users\victo\AppData\Local\Temp\ipykernel_22536\3986480264.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([d])], ignore_index=True)
C:\Users\victo\AppData\Local\Temp\ipykernel_22536\3986480264.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([d])], ignore_index=True)
C:\Users\victo\AppData\Local\Temp\ipykernel_22536\3986480264.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is

In [186]:
test_cvr = [33166966, 28142412, 70515113, 26366224, 58811211, 17045210, 27268188, 73648718, 61082913, 37537314, 15777249, 44581183, 45023117]

columns = ['entity', 'start date', 'end date', 'PnL', 'Assets', 'Equity', 'Number of employees']
num_rows = len(test_cvr)
zero_data = np.zeros((num_rows, len(columns)))
final_df = pd.DataFrame(zero_data, columns=columns)

for i in range(len(test_cvr)):
    data = offentliggoerelser_raw(str(test_cvr[i]))
    register2df(data)
    if len(register2df(data)) == 0:
        new_comp_dataframe = empty_df(test_cvr[i])
        final_df.iloc[i] = new_comp_dataframe.iloc[0]
    elif len(register2df(data)) == 1:
        cvrdata = cvr(test_cvr[i], ext="xml")
        doc = tojson(cvrdata)
        xmlasdf = json2df(doc)
        new_xmlasdf = xmlasdf[xmlasdf.columns.intersection(keep_cols)]
        df = new_xmlasdf
        new_comp_dataframe = new_dataframe(df)
        final_df.iloc[i] = new_comp_dataframe.iloc[0]
    else:
        cvrdata = cvr(test_cvr[i], ext="xml")
        doc = tojson(cvrdata)
        xmlasdf = json2df(doc)
        new_xmlasdf = xmlasdf[xmlasdf.columns.intersection(keep_cols)]
        new_xmlasdf['start_date'] = pd.to_datetime(new_xmlasdf['start_date'], format = '%Y-%m-%d', errors='coerce')
        latest_date = new_xmlasdf['start_date'].max()
        go_back_one_year = one_year_before_latest(latest_date)
        one_year_before_newest = latest_date - timedelta(days=go_back_one_year)
        filtered_new_xmlasdf = new_xmlasdf[new_xmlasdf['start_date'] > one_year_before_newest]
        df = filtered_new_xmlasdf
        new_comp_dataframe = new_dataframe(df)
        final_df.iloc[i] = new_comp_dataframe.iloc[0]
final_df


C:\Users\victo\AppData\Local\Temp\ipykernel_22536\3986480264.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([d])], ignore_index=True)
C:\Users\victo\AppData\Local\Temp\ipykernel_22536\3986480264.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([d])], ignore_index=True)
C:\Users\victo\AppData\Local\Temp\ipykernel_22536\3986480264.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is

,entity,start date,end date,PnL,Assets,Equity,Number of employees
0,33166966,2023-01-01 00:00:00,2023-12-31,-362354.0,2.901939e+07,1.960700e+05,14.0
1,28142412,2023-05-01 00:00:00,2024-04-30,338000000.0,5.510000e+09,2.981000e+09,1685.0
2,70515113,2024-01-01 00:00:00,2024-12-31,810484000.0,2.209418e+09,1.040602e+09,399.0
3,26366224,2023-01-01 00:00:00,2023-12-31,222522000.0,1.959799e+09,1.902990e+08,1394.0
4,58811211,2023-01-01 00:00:00,2023-12-31,177662199.0,2.065214e+09,9.674772e+08,1300.0
5,17045210,2023-01-01 00:00:00,2023-12-31,199968000.0,2.285059e+09,4.267460e+08,647.0
6,27268188,2024-01-01 00:00:00,2024-12-31,-33544000.0,2.443096e+09,1.183210e+08,1156.0
7,73648718,2023-01-01 00:00:00,2023-12-31,151034000.0,1.089191e+09,7.879120e+08,1170.0
8,61082913,2023-03-01 00:00:00,2024-02-29,206040000.0,3.069072e+09,2.100768e+09,739.0
9,37537314,2023-01-01 00:00:00,2023-12-31,150739775.0,1.769149e+09,1.001644e+09,765.0


In [190]:
final_df_test = final_df
len(final_df_test.columns)
# i = column, j = row
for i in range(len(final_df_test)):
    for j in range(len(final_df_test.columns)):
        if final_df_test.iloc[i, j] != final_df_test.iloc[i, j]:
            final_df_test.iloc[i,j] = 'test'
        if isinstance(final_df_test.iloc[i,j], float):
            final_df_test.iloc[i,j] = int(final_df_test.iloc[i,j])
        if isinstance(final_df_test.iloc[i,j], (pd.Timestamp, datetime)):
            final_df_test.iloc[i,j] = final_df_test.iloc[i,j].strftime("%Y-%m-%d")


final_df_test

,entity,start date,end date,PnL,Assets,Equity,Number of employees
0,33166966,2023-01-01,2023-12-31,-362354,29019390,196070,14
1,28142412,2023-05-01,2024-04-30,338000000,5510000000,2981000000,1685
2,70515113,2024-01-01,2024-12-31,810484000,2209418000,1040602000,399
3,26366224,2023-01-01,2023-12-31,222522000,1959799000,190299000,1394
4,58811211,2023-01-01,2023-12-31,177662199,2065214394,967477217,1300
5,17045210,2023-01-01,2023-12-31,199968000,2285059000,426746000,647
6,27268188,2024-01-01,2024-12-31,-33544000,2443096000,118321000,1156
7,73648718,2023-01-01,2023-12-31,151034000,1089191000,787912000,1170
8,61082913,2023-03-01,2024-02-29,206040000,3069072000,2100768000,739
9,37537314,2023-01-01,2023-12-31,150739775,1769149280,1001643766,765


In [192]:
data = offentliggoerelser_raw(33166966)
register2df(data)
cvrdata = cvr(33166966, ext="xml")
doc = tojson(cvrdata)
xmlasdf = json2df(doc)
new_xmlasdf = xmlasdf[xmlasdf.columns.intersection(keep_cols)]
new_xmlasdf
df = new_xmlasdf
new_comp_dataframe = new_dataframe(df)
new_xmlasdf['start_date'] = pd.to_datetime(new_xmlasdf['start_date'], format = '%Y-%m-%d', errors='coerce')
latest_date = new_xmlasdf['start_date'].max()
go_back_one_year = one_year_before_latest(latest_date)
one_year_before_newest = latest_date - timedelta(days=go_back_one_year)
filtered_new_xmlasdf = new_xmlasdf[new_xmlasdf['start_date'] > one_year_before_newest]
filtered_new_xmlasdf

C:\Users\victo\AppData\Local\Temp\ipykernel_22536\3986480264.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([d])], ignore_index=True)
C:\Users\victo\AppData\Local\Temp\ipykernel_22536\3986480264.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([d])], ignore_index=True)
C:\Users\victo\AppData\Local\Temp\ipykernel_22536\3986480264.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is

,entity,start_date,end_date,Equity,ProfitLoss,AverageNumberOfEmployees,Assets
12,33166966,2023-01-01,2023-01-01,500000.0,NaN,NaN,NaN
13,33166966,2023-01-01,2023-01-01,58424.0,NaN,NaN,NaN
14,33166966,2023-01-01,2023-01-01,3359015.0,NaN,NaN,NaN
15,33166966,2023-01-01,2023-12-31,NaN,-362354.0,14.0,NaN
16,33166966,2023-01-01,2023-12-31,NaN,-362354.0,NaN,NaN
17,33166966,2023-01-01,2023-12-31,NaN,NaN,NaN,NaN
18,33166966,2023-01-01,2023-12-31,NaN,2307309.0,NaN,NaN
19,33166966,2023-01-01,2023-12-31,NaN,NaN,NaN,NaN
20,33166966,2023-01-01,2023-12-31,NaN,NaN,NaN,NaN
21,33166966,2023-01-01,2023-12-31,NaN,NaN,NaN,NaN


In [193]:
data = offentliggoerelser_raw(44581183)
register2df(data)
cvrdata = cvr(44581183, ext="xml")
doc = tojson(cvrdata)
xmlasdf = json2df(doc)
new_xmlasdf = xmlasdf[xmlasdf.columns.intersection(keep_cols)]
new_xmlasdf

C:\Users\victo\AppData\Local\Temp\ipykernel_22536\3986480264.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([d])], ignore_index=True)
C:\Users\victo\AppData\Local\Temp\ipykernel_22536\3986480264.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([d])], ignore_index=True)
C:\Users\victo\AppData\Local\Temp\ipykernel_22536\3986480264.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is

,entity,start_date,end_date,Assets,ProfitLoss,Equity
0,44581183,2024-01-23,2024-09-30,54366.0,-40634.0,NaN
1,44581183,2024-01-23,2024-09-30,54366.0,NaN,NaN
2,44581183,2024-01-23,2024-09-30,54366.0,NaN,NaN
3,44581183,2024-01-23,2024-09-30,54366.0,NaN,NaN
4,44581183,2024-01-23,2024-09-30,54366.0,-40634.0,NaN
5,44581183,2024-09-30,2024-09-30,54366.0,NaN,-634.0
6,44581183,2024-09-30,2024-09-30,54366.0,NaN,40000.0
7,44581183,2024-09-30,2024-09-30,54366.0,NaN,-40634.0
